In [ ]:
# ========== 9. DISTILGPT2 (ИСПРАВЛЕННАЯ ВЕРСИЯ) ==========
print("\n" + "="*60)
print("ЭТАП 4: ИСПОЛЬЗОВАНИЕ DISTILGPT2")
print("="*60)

class DistilGPT2Comparer:
    def __init__(self):
        print("Загрузка DistilGPT2...")
        # Убираем framework="pt" - pipeline сам определяет
        self.generator = pipeline(
            "text-generation", 
            model="distilgpt2",
            device=0 if torch.cuda.is_available() else -1
        )
    
    def generate(self, prompt, max_length=20):
        result = self.generator(
            prompt,
            max_length=max_length,
            do_sample=True,
            top_k=50,
            temperature=0.8,
            pad_token_id=50256,
            truncation=True
        )
        return result[0]['generated_text']

# Инициализируем GPT2
gpt2 = DistilGPT2Comparer()

In [ ]:
# ========== 10. ROUGE МЕТРИКИ ДЛЯ DISTILGPT2 ==========
print("\n" + "="*60)
print("ЭТАП 4 (продолжение): ОЦЕНКА ROUGE ДЛЯ DISTILGPT2")
print("="*60)

def compute_rouge_gpt2(gpt2, test_texts, num_samples=50):
    """Оценка ROUGE для DistilGPT2"""
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2'], use_stemmer=True)
    rouge1_scores = []
    rouge2_scores = []
    
    # Берем случайные тексты
    samples = random.sample(test_texts, min(num_samples, len(test_texts)))
    
    for text in tqdm(samples, desc="Оценка GPT2"):
        words = text.split()
        if len(words) < 10:
            continue
        
        # Контекст (первые 70%) и референс (30%)
        split = int(len(words) * 0.7)
        context = ' '.join(words[:split])
        reference = ' '.join(words[split:])
        
        if len(reference.split()) < 2:
            continue
        
        # Генерация через GPT2
        try:
            generated = gpt2.generate(context, max_length=len(words))
            generated_words = generated.split()
            
            if len(generated_words) > split:
                generated_part = ' '.join(generated_words[split:split + len(reference.split())])
                
                # Считаем ROUGE
                scores = scorer.score(reference, generated_part)
                rouge1_scores.append(scores['rouge1'].fmeasure)
                rouge2_scores.append(scores['rouge2'].fmeasure)
        except Exception as e:
            continue
    
    return {
        'rouge1': np.mean(rouge1_scores) if rouge1_scores else 0,
        'rouge2': np.mean(rouge2_scores) if rouge2_scores else 0
    }

# Оцениваем GPT2
print("Оценка DistilGPT2 на тестовой выборке...")
gpt2_rouge = compute_rouge_gpt2(gpt2, test_texts[:500], num_samples=50)

print(f"\nGPT2 ROUGE-1: {gpt2_rouge['rouge1']:.4f}")
print(f"GPT2 ROUGE-2: {gpt2_rouge['rouge2']:.4f}")

In [ ]:

# ========== 11. СРАВНЕНИЕ ГЕНЕРАЦИИ ==========
print("\n" + "="*60)
print("ЭТАП 4 (продолжение): СРАВНЕНИЕ ГЕНЕРАЦИИ")
print("="*60)

test_prompts = [
    "i love",
    "today is",
    "i don't know",
    "the weather",
    "i think that",
    "good morning"
]

print("\nСРАВНЕНИЕ ПРИМЕРОВ ГЕНЕРАЦИИ:")
print("-" * 70)
print(f"{'Prompt':<20} {'LSTM':<30} {'GPT2':<30}")
print("-" * 70)

for prompt in test_prompts:
    # LSTM
    tokens = prompt.lower().split()
    indices = [vocab.get(t, vocab['<UNK>']) for t in tokens]
    indices = torch.tensor(indices).to(config.device)
    
    with torch.no_grad():
        lstm_gen = model.generate(indices, max_length=8, temperature=0.8)
    
    lstm_words = [idx_to_word.get(int(i), '<UNK>') for i in lstm_gen.cpu()]
    lstm_text = ' '.join(lstm_words)
    
    # GPT2
    gpt2_text = gpt2.generate(prompt, max_length=len(tokens) + 8)
    
    print(f"{prompt:<20} {lstm_text[:30]:<30} {gpt2_text[:30]:<30}")